# ROBERT Score Diagnosis (V1)

This notebook reads an extracted `run_context.json` and generates deterministic, auditable diagnosis outputs.

It does not call any external service and does not modify ROBERT core code.

Outputs written next to `run_context.json`:
- `diagnosis.json` (machine-readable flags and computed components)
- `diagnosis_summary.md` (short human-readable summary)

## Cell 2 Guide: Select a Run Context

Set `RUN_CONTEXT_PATH` to a specific file, or leave it as `None` to auto-select the newest `run_context.json` inside `agent/run_archive/`.

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import json

RUN_CONTEXT_PATH = None

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not infer project root from notebook working directory.")

PROJECT_ROOT = resolve_project_root()
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"

if RUN_CONTEXT_PATH is None:
    candidates = sorted(RUNS_ROOT.glob("**/run_context.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No run_context.json found. Run extract_context.ipynb first.")
    RUN_CONTEXT_PATH = candidates[0]
else:
    RUN_CONTEXT_PATH = Path(RUN_CONTEXT_PATH).resolve()

if not RUN_CONTEXT_PATH.exists():
    raise FileNotFoundError(f"run_context.json not found: {RUN_CONTEXT_PATH}")

context = json.loads(RUN_CONTEXT_PATH.read_text(encoding="utf-8"))
RUN_FOLDER = RUN_CONTEXT_PATH.parent

print(f"Loaded context: {RUN_CONTEXT_PATH}")
print(f"Prediction type: {context.get('pred_type')}")
print(f"Model: {context.get('ml_model')}")

Loaded context: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260515_173258__Hvapor/run_context.json
Prediction type: reg
Model: MVL


## Cell 4 Guide: Helper Functions

These are lightweight helpers for normalizing ROBERT-reported text fields and recording structured observations.

No ROBERT score is recomputed here.

In [11]:
def parse_ratio(text):
    """Split a ROBERT-reported ratio string like '106:3' into numeric values."""
    if not text or ":" not in str(text):
        return None, None
    left, right = str(text).split(":", 1)
    try:
        return float(left), float(right)
    except ValueError:
        return None, None


def normalize_verify_results(test_results):
    """Normalize VERIFY results into a stable list of {'test': ..., 'result': ...}.

    Supports list format like ['y_mean: PASSED', ...] and dict format when available.
    """
    normalized = []
    if isinstance(test_results, list):
        for item in test_results:
            if not isinstance(item, str) or ":" not in item:
                continue
            test_name, verdict = item.split(":", 1)
            normalized.append({"test": test_name.strip(), "result": verdict.strip()})
    elif isinstance(test_results, dict):
        for test_name, verdict in test_results.items():
            normalized.append({"test": str(test_name).strip(), "result": str(verdict).strip()})
    return normalized


def add_observation(obs_list, key, level, message, evidence):
    obs_list.append({
        "key": key,
        "level": level,
        "message": message,
        "evidence": evidence,
    })

## Cell 6 Guide: Build LLM-Ready Context From ROBERT Outputs

This cell only captures ROBERT-generated evidence from `run_context.json` and organizes it for downstream chat prompts.

It does not compute a new score, does not invent score components, and does not introduce new warning thresholds.

Any convenience math is labeled as derived and never presented as a ROBERT score substitute.

In [12]:
pred_type = context.get("pred_type")

# This artifact is designed as LLM-ready context built from ROBERT outputs.
# It must not introduce a competing score system.
diag = {
    "schema_version": "2.0",
    "diagnosed_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "results_dir": context.get("results_dir"),
    "pred_type": pred_type,
    "ml_model": context.get("ml_model"),
    "dataset_csv": context.get("dataset_csv"),
    "robert_command": context.get("robert_command"),
    "available": context.get("available", {}),
    "robert_score": context.get("score", {}),
    "evidence": {
        "predict": context.get("predict", {}),
        "verify": context.get("verify", {}),
        "curate": context.get("curate", {}),
        "parser_warnings": context.get("parser_warnings", []),
    },
    "observations": {
        "no_pfi": [],
        "pfi": [],
        "global": [],
    },
    "llm_context": {
        "prompt_intent": (
            "Use ROBERT outputs as authority. Explain implications and next checks "
            "without inventing new score systems."
        ),
        "report_view_mode": "User may view PDF while LLM uses extracted ROBERT context.",
        "source_policy": {
            "pdf_to_llm": False,
            "raw_text_context": True,
            "image_context": True,
        },
        "citation_fields": [
            "predict.no_pfi",
            "predict.pfi",
            "verify.no_pfi",
            "verify.pfi",
            "curate",
            "score",
        ],
    },
    "derived_values": {
        "convenience_only": True,
        "items": {
            "no_pfi": {},
            "pfi": {},
        },
    },
    "notes": [],
}

for variant in ["no_pfi", "pfi"]:
    p = context.get("predict", {}).get(variant, {})
    v = context.get("verify", {}).get(variant, {})
    obs = diag["observations"][variant]

    # VERIFY outcomes as ROBERT reported them
    verify_items = normalize_verify_results(v.get("test_results"))
    if verify_items:
        for item in verify_items:
            add_observation(
                obs,
                key=f"verify_{item['test']}",
                level=item["result"],
                message=f"ROBERT VERIFY: {item['test']} -> {item['result']}",
                evidence={
                    "source": f"verify.{variant}.test_results",
                    "test": item["test"],
                    "result": item["result"],
                },
            )
    else:
        add_observation(
            obs,
            key="verify_results_missing",
            level="info",
            message="VERIFY test results were not present in extracted ROBERT outputs.",
            evidence={"source": f"verify.{variant}.test_results"},
        )

    # ROBERT-reported aggregate VERIFY counts
    for name in ["failed_tests", "unclear_tests", "passed_tests", "flawed_mod_score"]:
        if v.get(name) is not None:
            add_observation(
                obs,
                key=f"verify_{name}",
                level="info",
                message=f"ROBERT reported {name} = {v.get(name)}.",
                evidence={"source": f"verify.{variant}.{name}", "value": v.get(name)},
            )

    # Data shape observations from ROBERT outputs
    for name in ["n_train", "n_test", "n_descriptors", "points_descp_ratio", "train_outlier_pct", "test_outlier_pct"]:
        if p.get(name) is not None:
            add_observation(
                obs,
                key=f"predict_{name}",
                level="info",
                message=f"ROBERT reported {name} = {p.get(name)}.",
                evidence={"source": f"predict.{variant}.{name}", "value": p.get(name)},
            )

    # Convenience derived value: train:descriptor numeric ratio from ROBERT counts
    n_train, n_desc = parse_ratio(p.get("points_descp_ratio"))
    if n_train is not None and n_desc not in (None, 0):
        derived_ratio = n_train / n_desc
        diag["derived_values"]["items"][variant]["train_to_descriptor_ratio"] = {
            "value": round(derived_ratio, 3),
            "from": f"predict.{variant}.points_descp_ratio",
            "label": "derived convenience value",
        }

# Global notes for LLM usage and evidence gaps
if not context.get("score") or all(v is None for v in context.get("score", {}).values()):
    diag["notes"].append(
        "ROBERT score not yet populated in run_context.json for this run. "
        "Use ROBERT report outputs as authoritative source when available."
    )

if context.get("parser_warnings"):
    add_observation(
        diag["observations"]["global"],
        key="parser_warnings_present",
        level="info",
        message="Extractor reported parser warnings; cite evidence carefully.",
        evidence={"source": "parser_warnings", "value": context.get("parser_warnings")},
    )

print("Built ROBERT-grounded context package (no score recomputation).")
for variant in ["no_pfi", "pfi"]:
    print(f"- {variant}: {len(diag['observations'][variant])} observations")

Built ROBERT-grounded context package (no score recomputation).
- no_pfi: 13 observations
- pfi: 13 observations


## Cell 8 Guide: Write Output Artifacts

This cell writes `diagnosis.json` and `diagnosis_summary.md` next to `run_context.json`.

The summary mirrors ROBERT-reported evidence and observations; it does not present a separate agent score.

In [13]:
diagnosis_path = RUN_FOLDER / "diagnosis.json"
summary_path = RUN_FOLDER / "diagnosis_summary.md"

diagnosis_path.write_text(json.dumps(diag, indent=2), encoding="utf-8")

lines = []
lines.append("# ROBERT Diagnosis Summary")
lines.append("")
lines.append(f"- Diagnosed at: {diag['diagnosed_at']}")
lines.append(f"- Prediction type: {diag.get('pred_type')}")
lines.append(f"- Model: {diag.get('ml_model')}")
lines.append(f"- Dataset: {diag.get('dataset_csv')}")
lines.append("")
lines.append("## Operating Mode")
lines.append("- ROBERT is authoritative for model scores and metrics.")
lines.append("- This artifact is extracted context for LLM explanation, not a competing score.")
lines.append("- PDF is for user display; LLM should use extracted raw text context and image references.")

score_block = diag.get("robert_score", {})
lines.append("")
lines.append("## ROBERT Score (As Extracted)")
if score_block and any(v is not None for v in score_block.values()):
    for key in ["no_pfi", "pfi"]:
        lines.append(f"- {key}: {score_block.get(key)}")
else:
    lines.append("- Not available in this run_context.json (null placeholders).")

for variant in ["no_pfi", "pfi"]:
    p = diag["evidence"].get("predict", {}).get(variant, {})
    v = diag["evidence"].get("verify", {}).get(variant, {})
    obs = diag["observations"].get(variant, [])

    lines.append("")
    lines.append(f"## {variant.upper()} (ROBERT-Reported Evidence)")

    if pred_type == "reg":
        lines.append(f"- R2 CV/Test: {p.get('r2_cv')} / {p.get('r2_test')}")
        lines.append(f"- RMSE CV/Test: {p.get('rmse_cv')} / {p.get('rmse_test')}")
        lines.append(f"- MAE CV/Test: {p.get('mae_cv')} / {p.get('mae_test')}")
    else:
        lines.append(f"- MCC CV/Test: {p.get('r2_cv')} / {p.get('r2_test')}")

    lines.append(f"- Train/Test points: {p.get('n_train')} / {p.get('n_test')}")
    lines.append(f"- Descriptors: {p.get('n_descriptors')} ({p.get('descriptors')})")
    lines.append(f"- Train:descriptor string: {p.get('points_descp_ratio')}")

    lines.append("")
    lines.append("### VERIFY")
    lines.append(f"- failed/unclear/passed: {v.get('failed_tests')} / {v.get('unclear_tests')} / {v.get('passed_tests')}")
    lines.append(f"- flawed_mod_score (ROBERT): {v.get('flawed_mod_score')}")
    lines.append(f"- test_results: {v.get('test_results')}")

    lines.append("")
    lines.append("### Observations")
    if not obs:
        lines.append("- No observations recorded.")
    for item in obs:
        lines.append(f"- [{item['level']}] {item['key']}: {item['message']}")

lines.append("")
lines.append("## Derived Convenience Values")
lines.append("- These are convenience transforms from ROBERT-reported fields and are not ROBERT score components.")
for variant in ["no_pfi", "pfi"]:
    derived = diag.get("derived_values", {}).get("items", {}).get(variant, {})
    lines.append(f"- {variant}: {derived}")

if diag.get("notes"):
    lines.append("")
    lines.append("## Notes")
    for note in diag["notes"]:
        lines.append(f"- {note}")

summary_path.write_text("\n".join(lines), encoding="utf-8")

print(f"Wrote: {diagnosis_path}")
print(f"Wrote: {summary_path}")
print()
print("Summary preview:")
print("\n".join(lines[:40]))

Wrote: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260515_173258__Hvapor/diagnosis.json
Wrote: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260515_173258__Hvapor/diagnosis_summary.md

Summary preview:
# ROBERT Diagnosis Summary

- Diagnosed at: 2026-05-15T21:39:43Z
- Prediction type: reg
- Model: MVL
- Dataset: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv

## Operating Mode
- ROBERT is authoritative for model scores and metrics.
- This artifact is extracted context for LLM explanation, not a competing score.
- PDF is for user display; LLM should use extracted raw text context and image references.

## ROBERT Score (As Extracted)
- Not available in this run_context.json (null placeholders).

## NO_PFI (ROBERT-Reported Evidence)
- R2 CV/Test: 0.77 / 0.78
- RMSE CV/Test: 5.7 / 5.5
- MAE CV/Test: 4.7 / 4.7
- Train/Test points: 106 / 27
- Descriptors: 3 (['Hardness', 'SGBP', 'g3'])
- Train:descriptor string: 106:3

### VERIFY
- faile

## Usage Notes

1. Run `agent/extract_context.ipynb` first to generate `run_context.json` from ROBERT outputs.
2. In Cell 3, set `RUN_CONTEXT_PATH` if you want a specific run; otherwise keep `None`.
3. Run all cells in order.
4. Inspect `diagnosis.json` and `diagnosis_summary.md` in the selected run folder.
5. This notebook builds LLM-ready context from ROBERT outputs; it does not compute an alternative score.
6. The user can read the PDF in the UI while the LLM uses extracted raw context and image references behind the scenes.